# 13 | More dispatch objectives: Mandi, SPO+, DBB, and an ensemble

**Purpose.** Notebook 12's window cross-entropy shows that dispatch alignment can move an LSTM from 4554 to 6301, and capture versus the oracle mean of 9000 rises by about one point. Cross-entropy pushes a single oracle window to rank 1. This notebook changes the surrogate, not the contract and not the LSTM head, and compares four losses from the literature plus an ensemble whose weights are chosen on the 30-day realized score.

**Inputs:**

- Frozen notebooks 11 and 12: `reports/holdout_rmse_v1/leaderboard.csv`, the v2 file, and `reports/holdout_dispatch_loss_v1/leaderboard.csv` plus predictions
- This trial: `reports/holdout_dfl_bakeoff_v1/` (run `experiments/holdout_dfl_bakeoff_v1/run.py` if it is missing)

**Five new recipes** (CompactDayLSTM and contextual features, unless the row is an ensemble):

1. `lstm_expected`: soft expected score, the non-one-hot version of cross-entropy
2. `lstm_pairwise`: Mandi pairwise
3. `lstm_spoplus`: SPO+ on the 3,321 power vectors, not the failed P01 calibrator elsewhere in the project
4. `lstm_dbb`: ICLR 2020 black-box differentiation through `optimize_day`, directly targeting realized score
5. `blend_val30`: Ridge + LightGBM + window-CE LSTM, simplex weights chosen on the 30-day realized score

**Check.** The headline comparison is predicted score, realized score, and capture. The line is notebook 12's `lstm_dispatch` at **6301.33 / 0.700**. Do not edit the Phase B frozen file, do not overwrite notebook 12's report directory, and do not treat the 59 days as a new champion selection.

**Protocol.** 271 fit days / select on 30-day realized score / refit on 301 / score the 59 days once. Seed 42.

The 59-day window was already used in notebooks 11 and 12. It is not an untouched holdout. Scores are raw dispatch scores, not currency.

Stored outputs were removed from this public copy. The executed charts used Chinese labels and cannot be redrawn without the raw series. Numbers in the text are the recorded results.


## Read this first

- **Relative to notebook 12's `lstm_dispatch`, SPO+ clears the line.** `lstm_spoplus` has 59-day realized score **6410**, capture **0.712**, predicted score **8529** (line 6301 / 0.700). This is an end-to-end LSTM, not the failed P01 calibrator.
- **Soft expected score does not beat the incumbent.** `lstm_expected` realized **6016** / 0.668, below window CE at 6301. The non-one-hot version is not better in this run.
- **Mandi pairwise collapses.** `lstm_pairwise` realized score is only **1000** / 0.111. The best 30-day grid point is 2312. The hinge flattens the curve.
- **DBB maximizes realized score directly, with a modest score and a forecast that barely commits.** `lstm_dbb` realized **5757**, predicted score only 154.
- **The ensemble puts 100% weight on the window-CE LSTM.** `blend_val30` weights `(ridge, lgb, lstm) = (0, 0, 1)`. The 59-day score is 5828, below the frozen notebook-12 LSTM. Weight selection did not add a new curve.
- **These 59 days are not a new champion selection.** The notebook only records which recipes cleared 6301.


In [ ]:
from pathlib import Path
import json
import os
import sys

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown
def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'phase_b').is_dir():
            return candidate
    raise FileNotFoundError('start this notebook from the repository root')


def show_figure(path: Path) -> None:
    display(Image(filename=str(path)))


REPO_ROOT = find_repo_root()
os.environ.setdefault('MPLCONFIGDIR', str(REPO_ROOT / '.matplotlib_cache'))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FIGURES = REPO_ROOT / 'outputs' / 'notebooks' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
REPORT = REPO_ROOT / 'reports' / 'holdout_dfl_bakeoff_v1'
REPORT12 = REPO_ROOT / 'reports' / 'holdout_dispatch_loss_v1'
PHASE_B = REPO_ROOT / 'reports' / 'phase_b' / 'final_metrics.json'
MONEY_LINE = 6301.331965
MONEY_CAPTURE = 0.700079
print({'repo_root': str(REPO_ROOT), 'figures': str(FIGURES.relative_to(REPO_ROOT))})


In [ ]:
board_path = REPORT / 'leaderboard.csv'
# Published copy is read-only: it never launches training. Frozen experiment outputs
# under reports/ are not distributed with this repository -- see DATA.md for how to
# regenerate them from the source data before running this notebook.
assert board_path.exists(), (
    f'missing {board_path}. This notebook reads frozen experiment outputs only; '
    'it does not train. See DATA.md to obtain the data and regenerate reports/.'
)
print('loading frozen trial artifacts from', REPORT.relative_to(REPO_ROOT))

trial = pd.read_csv(REPORT / 'leaderboard.csv')
ok = trial[trial['status'] == 'ok'].copy()
assert len(ok) == 5, f'expected 5 ok recipes, got {len(ok)}: {trial["status"].tolist()}'
selection = json.loads((REPORT / 'selection.json').read_text(encoding='utf-8'))
manifest = json.loads((REPORT / 'manifest.json').read_text(encoding='utf-8'))
assert manifest.get('holdout_not_used_for_selection') is True
assert manifest.get('does_not_overwrite_holdout_dispatch_loss_v1') is True
print(ok[['model', 'pred_profit_mean', 'realized_profit_mean', 'capture', 'beat_lstm_dispatch']].to_string(index=False))
print('phase_b frozen file still exists:', PHASE_B.exists())
print('nb12 report still exists:', (REPORT12 / 'leaderboard.csv').exists())


In [ ]:
v1 = pd.read_csv(REPO_ROOT / 'reports' / 'holdout_rmse_v1' / 'leaderboard.csv')
v2 = pd.read_csv(REPO_ROOT / 'reports' / 'holdout_rmse_v2' / 'leaderboard.csv')
nb12 = pd.read_csv(REPORT12 / 'leaderboard.csv')


def _row(source, recipe, pred, realized, capture, gap=None):
    return {
        'source': source,
        'recipe': recipe,
        'pred_profit_mean': float(pred),
        'realized_profit_mean': float(realized),
        'capture': float(capture),
        'profit_gap': float(pred - realized if gap is None else gap),
    }


def _from_board(frame, model, source):
    row = frame.loc[frame['model'] == model].iloc[0]
    return _row(source, f'{source}/{model}', row['pred_profit_mean'], row['realized_profit_mean'], row['capture'], row['profit_gap'])


anchors = [
    _from_board(v2, 'transformer_contextual', 'v2'),
    _from_board(v1, 'lstm_contextual', 'v1'),
    _from_board(nb12, 'lstm_dispatch', 'nb12'),
]
new_rows = []
for _, row in ok.iterrows():
    new_rows.append(_row('nb13', row['model'], row['pred_profit_mean'], row['realized_profit_mean'], row['capture'], row['profit_gap']))
compare = pd.DataFrame(anchors + new_rows)
best_new = ok.loc[ok['realized_profit_mean'].idxmax()]
beat = float(best_new['realized_profit_mean']) > MONEY_LINE
if beat:
    verdict = (
        f"Relative to notebook 12's lstm_dispatch, a new dispatch surrogate clears the line. "
        f"The best is {best_new['model']}: realized {float(best_new['realized_profit_mean']):.0f}, "
        f"capture {float(best_new['capture']):.3f} (line 6301 / 0.700)."
    )
else:
    verdict = (
        f"This trial does not beat notebook 12's lstm_dispatch. "
        f"The best new recipe is {best_new['model']}: realized {float(best_new['realized_profit_mean']):.0f}, "
        f"capture {float(best_new['capture']):.3f}, against 6301 / 0.700."
    )
verdict += ' These 59 days are not a new champion selection.'
display(Markdown(f'**Result:** {verdict}'))
print(verdict)


## Headline: predicted score, realized score, capture

The line is drawn at `lstm_dispatch` 6301. RMSE is not the decision.


In [ ]:
show = compare.copy()
show['pred_profit'] = show['pred_profit_mean'].round(0).astype(int)
show['realized'] = show['realized_profit_mean'].round(0).astype(int)
show['capture'] = show['capture'].round(3)
show['gap'] = show['profit_gap'].round(0).astype(int)
show['beat_6301'] = show['realized_profit_mean'] > MONEY_LINE
display(
    show[['source', 'recipe', 'pred_profit', 'realized', 'capture', 'gap', 'beat_6301']]
    .sort_values('realized', ascending=False)
    .style.format({'capture': '{:.3f}'})
    .hide(axis='index')
)


In [ ]:
order = compare.sort_values('realized_profit_mean', ascending=False)
colors = ['#2a6f97' if src == 'nb13' else '#c44536' if src == 'nb12' else '#6c757d' for src in order['source']]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
axes[0].barh(order['recipe'], order['realized_profit_mean'], color=colors)
axes[0].axvline(MONEY_LINE, color='#111', linestyle='--', linewidth=1, label='nb12 lstm_dispatch 6301')
axes[0].set_xlabel('realized mean')
axes[0].set_title('59-day realized')
axes[0].invert_yaxis()
axes[0].legend(loc='lower right', fontsize=8)
axes[1].barh(order['recipe'], order['capture'], color=colors)
axes[1].axvline(MONEY_CAPTURE, color='#111', linestyle='--', linewidth=1, label='0.700')
axes[1].set_xlabel('capture')
axes[1].set_title('59-day capture')
axes[1].invert_yaxis()
axes[1].legend(loc='lower right', fontsize=8)
fig.suptitle('Notebook 13 vs 12 incumbent · blue = new surrogate', fontsize=11)
fig.tight_layout()
p1 = FIGURES / 'nb13_fig1_realized_capture.png'
fig.savefig(p1, dpi=140, bbox_inches='tight')
plt.close(fig)
print(p1.relative_to(REPO_ROOT))
show_figure(p1)


In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 5.2))
for src, sub in compare.groupby('source'):
    marker = 'o' if src == 'nb13' else 'D' if src == 'nb12' else 's'
    ax.scatter(sub['pred_profit_mean'], sub['realized_profit_mean'], s=70, marker=marker, label=src)
    for _, row in sub.iterrows():
        ax.annotate(str(row['recipe']).split('/')[-1], (row['pred_profit_mean'], row['realized_profit_mean']), fontsize=7, xytext=(4, 4), textcoords='offset points')
ax.axhline(MONEY_LINE, color='#111', linestyle='--', linewidth=1)
ax.set_xlabel('pred_profit mean')
ax.set_ylabel('realized mean')
ax.set_title('Honesty: predicted vs realized')
ax.legend(fontsize=8)
fig.tight_layout()
p2 = FIGURES / 'nb13_fig2_pred_vs_realized.png'
fig.savefig(p2, dpi=140, bbox_inches='tight')
plt.close(fig)
print(p2.relative_to(REPO_ROOT))
show_figure(p2)


## 2025-12-12: the best new model against notebook 12's LSTM


In [ ]:
from src.phase_b.dispatch import optimize_day
from src.phase_b.evaluation import score_day

old_pred = pd.read_parquet(REPORT12 / 'predictions.parquet')
new_pred = pd.read_parquet(REPORT / 'predictions.parquet')
best_name = str(best_new['model'])
day = pd.Timestamp('2025-12-12')
old_day = old_pred.loc[pd.to_datetime(old_pred['times']).dt.normalize() == day]
new_day = new_pred.loc[pd.to_datetime(new_pred['times']).dt.normalize() == day]
actual = old_day['A'].to_numpy(dtype=float)
old = old_day['lstm_dispatch'].to_numpy(dtype=float)
new = new_day[best_name].to_numpy(dtype=float)
slots = np.arange(96)
old_dec = optimize_day(old)
new_dec = optimize_day(new)
ora = optimize_day(actual)
fig, ax = plt.subplots(figsize=(11, 4.2))
ax.plot(slots, actual, color='#111', linewidth=1.6, label='true A')
ax.plot(slots, old, color='#c44536', linewidth=1.4, label='nb12 lstm_dispatch')
ax.plot(slots, new, color='#2a6f97', linewidth=1.6, label=best_name)
if ora.tc is not None:
    ax.axvspan(ora.tc, ora.tc + 8, color='#6c757d', alpha=0.15)
    ax.axvspan(ora.td, ora.td + 8, color='#6c757d', alpha=0.28)
ax.set_xlabel('15-min slot')
ax.set_ylabel('price A')
ax.set_title(
    f'2025-12-12  lstm_dispatch {score_day(actual, old_dec.power):.0f}  ·  '
    f'{best_name} {score_day(actual, new_dec.power):.0f}  ·  oracle {score_day(actual, ora.power):.0f}'
)
ax.legend(ncol=3, fontsize=8)
fig.tight_layout()
p3 = FIGURES / 'nb13_fig3_dec12.png'
fig.savefig(p3, dpi=140, bbox_inches='tight')
plt.close(fig)
print(p3.relative_to(REPO_ROOT))
show_figure(p3)


## 30-day selection grid

The 59-day window is not used for this grid.


In [ ]:
grid_rows = []
for kind, payload in selection.items():
    chosen = payload.get('params', {})
    for point in payload.get('grid', []):
        if kind == 'blend':
            label = f"blend r{point.get('w_ridge')} l{point.get('w_lgb')} m{point.get('w_lstm')}"
            realized = point.get('val30_realized')
            is_chosen = (
                abs(point.get('w_ridge', -1) - chosen.get('w_ridge', -2)) < 1e-9
                and abs(point.get('w_lgb', -1) - chosen.get('w_lgb', -2)) < 1e-9
            )
        else:
            params = point.get('params', point)
            label = f"{kind} {json.dumps(params, sort_keys=True)}"
            realized = point.get('val30_realized')
            is_chosen = params == chosen and point.get('status', 'ok') == 'ok'
        grid_rows.append({'kind': kind, 'label': label, 'val30_realized': realized, 'chosen': is_chosen})
grid = pd.DataFrame(grid_rows)
grid = grid[grid['val30_realized'].notna()].copy()
display(grid.sort_values(['kind', 'val30_realized'], ascending=[True, False]).head(24).style.hide(axis='index'))

plot = grid.copy()
if len(plot) > 24:
    plot = pd.concat([
        plot[plot['chosen']],
        plot.sort_values('val30_realized', ascending=False).head(18),
    ]).drop_duplicates(subset=['label'])
plot = plot.sort_values('val30_realized', ascending=False)
fig, ax = plt.subplots(figsize=(10, 4.2))
bar_colors = ['#2a6f97' if chosen else '#adb5bd' for chosen in plot['chosen']]
ax.bar(np.arange(len(plot)), plot['val30_realized'].astype(float), color=bar_colors)
ax.set_xticks(np.arange(len(plot)))
ax.set_xticklabels(plot['label'], rotation=45, ha='right', fontsize=7)
ax.set_ylabel('30-day realized')
ax.set_title('Selection grid · blue = chosen · holdout unused')
fig.tight_layout()
p4 = FIGURES / 'nb13_fig4_val30_grid.png'
fig.savefig(p4, dpi=140, bbox_inches='tight')
plt.close(fig)
print(p4.relative_to(REPO_ROOT))
show_figure(p4)


## These 59 days are not a new champion selection

The window was already used in notebooks 11 and 12. This notebook only asks whether replacing single-window cross-entropy with Mandi, SPO+, DBB, or an ensemble beats `lstm_dispatch`. It does not refresh the leaderboard, does not edit Phase B, and does not retune on the 59 days. The SPO+ row is an end-to-end LSTM, not the failed P01 calibrator in `decision_guided_agent_v1`.
